In [1]:
# Install required packages for reading BXSF files and 3D plotting
import subprocess
import sys

packages = ['plotly', 'pymatgen', 'scikit-image']
for package in packages:
    try:
        __import__(package)
        print(f"{package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

plotly is already installed
pymatgen is already installed
Installing scikit-image...


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import re
import os

In [3]:
import numpy as np
import plotly.graph_objects as go
from skimage import measure

def read_bxsf_fermi_surface(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        if "Fermi Energy" in line:
            fermi_energy = float(line.strip().split()[-1])
            break

    band_block_start = lines.index('  BEGIN_BANDGRID_3D_fermi\n')
    num_bands = int(lines[band_block_start + 1].strip())
    nx, ny, nz = map(int, lines[band_block_start + 2].strip().split())

    band_data = []
    index = band_block_start + 7

    for band in range(num_bands):
        while index < len(lines) and not lines[index].strip().startswith('BAND:'):
            index += 1
        index += 1

        values = []
        while index < len(lines):
            line = lines[index].strip()
            if line.startswith('BAND:') or line.startswith('END'):
                break
            values.extend([float(x) for x in line.split()])
            index += 1

        band_data.append(np.array(values).reshape((nx, ny, nz)))

    return np.array(band_data), fermi_energy, (nx, ny, nz)

# === Usage ===
band_data_up, fermi_energy_up, grid_shape_up = read_bxsf_fermi_surface("RuO2_up.bxsf")
band_data_down, fermi_energy_down, grid_shape_down = read_bxsf_fermi_surface("RuO2_dn.bxsf")

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
from skimage import measure

# ====== Load both spin channels ======
# Make sure you have your working read_bxsf_fermi_surface function defined before this.
band_data_up, fermi_energy_up, grid_shape_up = read_bxsf_fermi_surface("RuO2_up.bxsf")
band_data_down, fermi_energy_down, grid_shape_down = read_bxsf_fermi_surface("RuO2_dn.bxsf")

# ====== Create subplots ======
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]],horizontal_spacing=0.01,
                    subplot_titles=("Spin Up", "Spin Down"))

def add_bz_box(fig, row, col, box_min=-0.5, box_max=0.5):
    """Draws a cube around the BZ."""
    edges = [
        [(box_min, box_min, box_min), (box_max, box_min, box_min)],
        [(box_min, box_min, box_min), (box_min, box_max, box_min)],
        [(box_min, box_min, box_min), (box_min, box_min, box_max)],
        [(box_max, box_min, box_min), (box_max, box_max, box_min)],
        [(box_max, box_min, box_min), (box_max, box_min, box_max)],
        [(box_min, box_max, box_min), (box_max, box_max, box_min)],
        [(box_min, box_max, box_min), (box_min, box_max, box_max)],
        [(box_min, box_min, box_max), (box_max, box_min, box_max)],
        [(box_min, box_min, box_max), (box_min, box_max, box_max)],
        [(box_max, box_max, box_min), (box_max, box_max, box_max)],
        [(box_max, box_min, box_max), (box_max, box_max, box_max)],
        [(box_min, box_max, box_max), (box_max, box_max, box_max)],
    ]
    for start, end in edges:
        fig.add_trace(
            go.Scatter3d(
                x=[start[0], end[0]],
                y=[start[1], end[1]],
                z=[start[2], end[2]],
                mode="lines",
                line=dict(color="black", width=4),
                showlegend=False
            ),
            row=row, col=col
        )

# ====== Colors for each band ======
band_colors = {
    15: "teal",
    16: "orange"
}

# ====== Plot both spin channels ======
for spin, (band_data, fermi_energy, grid_shape, col, title) in enumerate([
    (band_data_up, fermi_energy_up, grid_shape_up, 1, "Spin Up"),
    (band_data_down, fermi_energy_down, grid_shape_down, 2, "Spin Down")
]):
    nx, ny, nz = grid_shape
    repeat = 3
    for idx in [15, 16]:  # Band indices
        surface = band_data[idx] - fermi_energy
        surface_tiled = np.tile(surface, (repeat, repeat, repeat))
        nx_t, ny_t, nz_t = surface_tiled.shape

        verts, faces, _, _ = measure.marching_cubes(surface_tiled, level=0.0)
        verts[:, 0] = verts[:, 0] / (nx_t - 1) * (repeat * 1.0)
        verts[:, 1] = verts[:, 1] / (ny_t - 1) * (repeat * 1.0)
        verts[:, 2] = verts[:, 2] / (nz_t - 1) * (repeat * 1.0)

        xv, yv, zv = verts.T
        xv = xv / (repeat * 1.0) * 2.0 - 1.0
        yv = yv / (repeat * 1.0) * 2.0 - 1.0
        zv = zv / (repeat * 1.0) * 2.0 - 1.0

        i_faces, j_faces, k_faces = faces.T

        mesh = go.Mesh3d(
            x=xv, y=yv, z=zv,
            i=i_faces, j=j_faces, k=k_faces,
            opacity=0.6,
            color=band_colors[idx],
            name=f'Band {idx + 1}',
            showscale=False
        )
        fig.add_trace(mesh, row=1, col=col)

    # Add BZ cube
    add_bz_box(fig, row=1, col=col)

    # Update scene with larger fonts
    fig.update_scenes(
        xaxis=dict(
            title=dict(text='kx', font=dict(size=18)),
            range=[-0.5, 0.5],
            tickvals=[-0.5, 0, 0.5],
            tickfont=dict(size=14)
        ),
        yaxis=dict(
            title=dict(text='ky', font=dict(size=18)),
            range=[-0.5, 0.5],
            tickvals=[-0.5, 0, 0.5],
            tickfont=dict(size=14)
        ),
        zaxis=dict(
            title=dict(text='kz', font=dict(size=18)),
            range=[-0.5, 0.5],
            tickvals=[-0.5, 0, 0.5],
            tickfont=dict(size=14)
        ),
        aspectmode='cube',
        row=1, col=col
    )

fig.update_layout(
    margin=dict(l=5, r=5, b=10, t=30),
    width=1000,
    height=500
)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go
# Define corners of the rectangular BZ in fractional coordinates
a=1.0
b=1.0
c=1.0
X = np.array([-0.5, 0.5])
Y = np.array([-0.5, 0.5])
Z = np.array([-0.5, 0.5])

# Scale to reciprocal space units
X *= a
Y *= b
Z *= c

# BZ edges (each pair is a segment)
edges = []
for x in X:
    for y in Y:
        edges.append(((x, y, Z[0]), (x, y, Z[1])))
for x in X:
    for z in Z:
        edges.append(((x, Y[0], z), (x, Y[1], z)))
for y in Y:
    for z in Z:
        edges.append(((X[0], y, z), (X[1], y, z)))

# Create edge traces
edge_traces = []
for start, end in edges:
    edge_traces.append(
        go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='black', width=10),
            showlegend=False
        )
    )
# High symmetry points in fractional coordinates
high_sym_points = {
    "Γ": (0, 0, 0),
    "X": (0.5*a, 0, 0),
    "Y": (0, 0.5*b, 0),
    "Z": (0, 0, 0.5*c),
    "M": (0.5*a, 0.5*b, 0),
    "A": (0.5*a, 0.5*b, 0.5*c),
    "R": (0.5*a, 0, 0.5*c),
    "M'": (-0.5*a, 0.5*b, 0),
    "A'": (-0.5*a, 0.5*b, 0.5*c),
    "R'": ( 0, 0.5*b, 0.5*c)
}
# Scatter for high symmetry points
sym_trace = go.Scatter3d(
    x=[v[0] for v in high_sym_points.values()],
    y=[v[1] for v in high_sym_points.values()],
    z=[v[2] for v in high_sym_points.values()],
    mode='markers+text',
    marker=dict(size=15, color='red'),
    text=list(high_sym_points.keys()),
    textposition='top left',
    showlegend=False,
    textfont=dict(
        size=30,          # bigger font size
        color='black',    # text color
        family='Arial Black, Arial, sans-serif',  # boldish font family
    ),
)

axis_lines = [
    go.Scatter3d(x=[-0.5*a, 0.5*a], y=[0, 0], z=[0, 0], mode='lines',        line=dict(color='blue', width=4), name='kx axis'),
    go.Scatter3d(x=[0, 0], y=[-0.5*b, 0.5*b], z=[0, 0], mode='lines',       line=dict(color='green', width=4), name='ky axis'),
    go.Scatter3d(x=[0, 0], y=[0, 0], z=[-0.5*c, 0.5*c], mode='lines',         line=dict(color='red', width=4), name='kz axis'),
    go.Scatter3d(x=[0, 0], y=[0.5*b,0.5*b], z =[0*c, 0.5*c], mode='lines',    line=dict(color='red', width=4), name='kz axis'),
    go.Scatter3d(x=[0.5*a, 0.5*a], y=[0, 0], z=[0*c, 0.5*c], mode='lines',    line=dict(color='red', width=4), name='kz axis'),
    go.Scatter3d(x=[0.5*a, 0.5*a], y=[0, 0.5*b], z=[0*c, 0*c], mode='lines',line=dict(color='green', width=4), name='kz axis'),
    go.Scatter3d(x=[0, 0.5*b], y=[0.5*b, 0.5*b], z=[0*c, 0*c], mode='lines', line=dict(color='blue', width=4), name='kz axis')
]
fig = go.Figure(data=edge_traces + axis_lines + [sym_trace])
# Axis labels
fig.update_layout(
    scene=dict(
        xaxis=dict(
            title=dict(text='kx', font=dict(size=22)),
            tickvals=[-0.5*a, 0, 0.5*a],
            ticktext=['-0.5', '0', '0.5'],
            tickfont=dict(size=20, family='Arial, sans-serif'),
        ),
        yaxis=dict(
            title=dict(text='ky', font=dict(size=22)),
            tickvals=[-0.5*b, 0],
            ticktext=['-0.5', '0'],
            tickfont=dict(size=20, family='Arial, sans-serif'),
        ),
        zaxis=dict(
            title=dict(text='kz', font=dict(size=22)),
            tickvals=[0, 0.5*c],
            ticktext=['0', '0.5'],
            tickfont=dict(size=20, family='Arial, sans-serif'),
        ),
        aspectmode='cube'
    ),
    showlegend=False
)

fig.update_layout(
width=800,
height=800, margin=dict(l=10, r=10, t=10, b=10),  # Reduce margins
)
fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go

# =====================
# Unit cell parameters
# =====================
a = 4.5405
c = 3.1323
a1 = np.array([a, 0, 0])
a2 = np.array([0, a, 0])
a3 = np.array([0, 0, c])

# =====================
# Atomic positions (fractional)
# =====================
atoms = [
    ("Ru1", np.array([0.00000, 0.00000, 0.00000])),
    ("Ru2", np.array([0.50000, 0.50000, 0.50000])),
    ("O",  np.array([0.19500, 0.80500, 0.50000])),
    ("O",  np.array([0.80500, 0.19400, 0.50000])),
    ("O",  np.array([0.69500, 0.69500, 0.00000])),
    ("O",  np.array([0.30500, 0.30500, 0.00000])),
]

# Colors and radii
colors = {"Ru1": "blue", "Ru2": "red", "O": "gray"}
radii = {"Ru1": 0.4, "Ru2": 0.4, "O": 0.25}

# =====================
# Expand atoms with periodic translations
# =====================
expanded_atoms = []
for name, pos in atoms:
    for i in [-1, 0, 1]:
        for j in [-1, 0, 1]:
            for k in [-1, 0, 1]:
                new_frac = pos + np.array([i, j, k])
                cart = new_frac[0] * a1 + new_frac[1] * a2 + new_frac[2] * a3
                if (0 - radii[name] <= cart[0] <= a + radii[name] and
                    0 - radii[name] <= cart[1] <= a + radii[name] and
                    0 - radii[name] <= cart[2] <= c + radii[name]):
                    expanded_atoms.append((name, cart))

# =====================
# Generate spheres
# =====================
def create_sphere(center, radius, color):
    u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
    x = radius * np.cos(u) * np.sin(v) + center[0]
    y = radius * np.sin(u) * np.sin(v) + center[1]
    z = radius * np.cos(v) + center[2]
    return go.Surface(
        x=x, y=y, z=z,
        colorscale=[[0, color], [1, color]],
        showscale=False,
        opacity=1.0
    )

# =====================
# Bonds (Ru–O only)
# =====================
bond_cutoff = 2.2
bonds = []
for i, (name1, pos1) in enumerate(expanded_atoms):
    if name1 not in ["Ru1", "Ru2"]:
        continue
    for j, (name2, pos2) in enumerate(expanded_atoms):
        if name2 != "O":
            continue
        dist = np.linalg.norm(pos1 - pos2)
        if dist <= bond_cutoff:
            bonds.append((pos1, pos2))

bond_traces = []
for p1, p2 in bonds:
    bond_traces.append(go.Scatter3d(
        x=[p1[0], p2[0]],
        y=[p1[1], p2[1]],
        z=[p1[2], p2[2]],
        mode="lines",
        line=dict(color="black", width=5),
        showlegend=False
    ))

# =====================
# Create figure
# =====================
fig = go.Figure()

# Add atoms
for name, pos in expanded_atoms:
    fig.add_trace(create_sphere(pos, radii[name], colors[name]))

# Add bonds
for trace in bond_traces:
    fig.add_trace(trace)

# Unit cell edges
edges = [
    [(0, 0, 0), (a, 0, 0)],
    [(0, 0, 0), (0, a, 0)],
    [(0, 0, 0), (0, 0, c)],
    [(a, 0, 0), (a, a, 0)],
    [(a, 0, 0), (a, 0, c)],
    [(0, a, 0), (a, a, 0)],
    [(0, a, 0), (0, a, c)],
    [(0, 0, c), (a, 0, c)],
    [(0, 0, c), (0, a, c)],
    [(a, a, 0), (a, a, c)],
    [(a, 0, c), (a, a, c)],
    [(0, a, c), (a, a, c)],
]
for start, end in edges:
    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]],
        y=[start[1], end[1]],
        z=[start[2], end[2]],
        mode="lines",
        line=dict(color="black", width=3),
        showlegend=False
    ))

# Padding so spheres aren't cut off
pad_x = max(radii.values())
pad_y = max(radii.values())
pad_z = max(radii.values())

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[-pad_x, a + pad_x], showticklabels=False, title=""),
        yaxis=dict(range=[-pad_y, a + pad_y], showticklabels=False, title=""),
        zaxis=dict(range=[-pad_z, c + pad_z], showticklabels=False, title="")
    ),
    width=800,
    height=600,
    margin=dict(l=10, r=10, t=10, b=10),
)

# =====================
# Legend (round markers like balls)
# =====================
for atom_type in ["Ru1", "Ru2", "O"]:
    fig.add_trace(go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode='markers',
        marker=dict(size=12, color=colors[atom_type], symbol='circle'),
        name=atom_type,
        showlegend=True
    ))

fig.update_layout(
    showlegend=True,
    legend=dict(
        x=0.8, y=0.88,
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="black",
        borderwidth=1
    )
)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go
# Define corners of the rectangular BZ in fractional coordinates
a=1.0
b=1.0
c=1.0
X = np.array([-0.5, 0.5])
Y = np.array([-0.5, 0.5])
Z = np.array([-0.5, 0.5])

# Scale to reciprocal space units
X *= a
Y *= b
Z *= c

# BZ edges (each pair is a segment)
edges = []
for x in X:
    for y in Y:
        edges.append(((x, y, Z[0]), (x, y, Z[1])))
for x in X:
    for z in Z:
        edges.append(((x, Y[0], z), (x, Y[1], z)))
for y in Y:
    for z in Z:
        edges.append(((X[0], y, z), (X[1], y, z)))

# Create edge traces
edge_traces = []
for start, end in edges:
    edge_traces.append(
        go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='black', width=10),
            showlegend=False
        )
    )
# High symmetry points in fractional coordinates
# High symmetry points in fractional coordinates (list instead of dict)
high_sym_points = [
    ("Γ", (0, 0, 0)),
    ("Z", (0, 0, 0.5*c)),
    ("Z", (0, 0, -0.5*c)),
    ("M", (0.5*a, 0.5*b, 0)),
    ("M", (-0.5*a, -0.5*b, 0)),
    ("M", (0.5*a, 0.5*b, 0)),
    ("A", (0.5*a, 0.5*b, 0.5*c)),
    ("A", (0.5*a, 0.5*b, -0.5*c)),
    ("A", (-0.5*a, -0.5*b, 0.5*c)),
    ("A", (-0.5*a, -0.5*b, -0.5*c)),
]

# Scatter for high symmetry points
sym_trace = go.Scatter3d(
    x=[v[0] for _, v in high_sym_points],
    y=[v[1] for _, v in high_sym_points],
    z=[v[2] for _, v in high_sym_points],
    mode='markers+text',
    marker=dict(size=15, color='red'),
    text=[name for name, _ in high_sym_points],
    textposition='top left',
    showlegend=False,
    textfont=dict(
        size=30,
        color='black',
        family='Arial Black, Arial, sans-serif',
    ),
)


axis_lines = [
    go.Scatter3d(x=[-0.5*a, 0.5*a], y=[-0.5*b, 0.5*b], z=[-0.5*c, -0.5*c], mode='lines',line=dict(color='black', width=10)),
    go.Scatter3d(x=[-0.5*a, 0.5*a], y=[-0.5*b, 0.5*b], z=[0.5*c, 0.5*c], mode='lines', line=dict(color='black', width=10)),
]
fig = go.Figure(data=edge_traces + axis_lines + [sym_trace])
# Axis labels
fig.update_layout(
    scene=dict(
        xaxis=dict(
            title=dict(text='kx', font=dict(size=22)),
            tickvals=[-0.5*a, 0, 0.5*a],
            ticktext=['-0.5', '0', '0.5'],
            tickfont=dict(size=20, family='Arial, sans-serif'),
        ),
        yaxis=dict(
            title=dict(text='ky', font=dict(size=22)),
            tickvals=[-0.5*b, 0],
            ticktext=['-0.5', '0'],
            tickfont=dict(size=20, family='Arial, sans-serif'),
        ),
        zaxis=dict(
            title=dict(text='kz', font=dict(size=22)),
            tickvals=[0, 0.5*c],
            ticktext=['0', '0.5'],
            tickfont=dict(size=20, family='Arial, sans-serif'),
        ),
        aspectmode='cube'
    ),
    showlegend=False
)
# Coordinates of a diagonal plane (example: y = x, spanning z)
plane_x = [-0.5*a, 0.5*a, 0.5*a, -0.5*a]
plane_y = [-0.5*b, 0.5*b, 0.5*b, -0.5*b]
plane_z = [-0.5*c, -0.5*c, 0.5*c, 0.5*c]

# Triangles for the mesh (two triangles make a square)
i_faces = [0, 0]
j_faces = [1, 2]
k_faces = [2, 3]

plane_trace = go.Mesh3d(
    x=plane_x,
    y=plane_y,
    z=plane_z,
    i=i_faces,
    j=j_faces,
    k=k_faces,
    opacity=0.3,
    color='blue',
    showscale=False
)

# Add to your figure
fig.add_trace(plane_trace)
fig.update_layout(
width=800,
height=800, margin=dict(l=10, r=10, t=10, b=10),  # Reduce margins
)
fig.show()
